# Etapa 5m — ¿Hacen falta todos los mapas? Selección con L1

**Ronda de mejoras para la entrega final (desvío 14.11 del plan).** Solo desarrollo, la misma validación cruzada por personas. El conjunto de prueba no se usa.

**Pregunta.** Con las rondas anteriores el algoritmo llegó a 31 mapas, varios agregados con efectos mínimos (las aperturas por reconstrucción, +0,001 de Matthews; B-COSFIRE, +0,2 puntos de vasos finos). Los mapas están muy correlacionados (el mismo filtro en 5 escalas más la mayor respuesta), y la regresión sin restricción reparte el peso entre mapas parecidos, a veces con signos opuestos que se compensan. ¿Cuáles hacen falta?

**Método** (`stats.fit_logistic_l1`, `pipeline.fit_weights_l1`). Regresión logística con penalidad L1 (lasso; Tibshirani, 1996): se minimiza la devianza + λ · Σ |w_j|, lo mismo que maximizar la verosimilitud con la restricción Σ |w_j| ≤ t. La penalidad deja en cero exactamente los pesos de los mapas que no agregan información propia. Se resuelve con Newton proximal y descenso por coordenadas (Friedman, Hastie y Tibshirani, 2010). **λ se elige dentro del entrenamiento de cada vuelta:** validación cruzada con los 4 grupos de personas de entrenamiento, devianza en los píxeles del grupo de afuera, y la regla de un error estándar (Hastie, Tibshirani y Friedman, 2009, sección 7.10): el λ más grande, es decir el modelo más simple, cuya devianza no se distingue de la mínima. El valor fijo del umbral se elige después, como siempre (`choose_value`).

**Candidatas** (algoritmo actual: Wiener con ruido estimado, relleno de exudados, división, entrenamiento equilibrado por grosor, valor fijo):
- **A.** Los 31 mapas, regresión sin restricción (el algoritmo actual).
- **B.** L1 sobre los mismos 31 mapas. Solo puede sacar mapas.
- **C.** L1 sobre los 49 mapas de `05l` (los 31 más simetría de fase, aperturas por caminos y autovalores del hessiano). **Es descriptiva y no se puede adoptar:** muestra qué mapas elegiría L1 si pudiera elegir entre todos los probados.

**Medidas**, en las personas de afuera, a resolución original, en el campo de visión de la base: Matthews, vasos finos (sensibilidad de línea central), Dice de línea central y precisión; cuántos mapas usa cada vuelta y cuáles.

**Regla de adopción (fijada antes de correr).** B reemplaza a A si
1. no es inferior a A en Matthews, vasos finos ni Dice de línea central (límite inferior > −0,03; la regla de la más simple entre las no inferiores, sección 5.3 del esquema), **y**
2. usa menos mapas que A en las 5 vueltas.

Si se adopta, los mapas que quedan en cero dejan de calcularse, y la lista final sale de ajustar L1 con las 30 imágenes al cerrar la versión 2. Si no, quedan los 31.

**Control:** A tiene que reproducir el Matthews de C en `05l` (0,7527).

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 80)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
MODE = C.SEGMENT_BACKGROUND_MODE
SUPER = C.SEGMENT_FEATURES + [n for names in C.NEW_FEATURE_FAMILIES.values() for n in names
                              if n not in C.SEGMENT_FEATURES]
CANDS = {"A. 31 mapas": C.SEGMENT_FEATURES, "B. L1 sobre los 31": C.SEGMENT_FEATURES, "C. L1 sobre los 49": SUPER}
COLS = {c: [SUPER.index(n) for n in names] for c, names in CANDS.items()}
ORDER = list(CANDS)
WORKERS = 4
print(MODE, "|", {c: len(v) for c, v in CANDS.items()})

división + relleno de exudados + Wiener con ruido estimado | {'A. 31 mapas': 31, 'B. L1 sobre los 31': 31, 'C. L1 sobre los 49': 49}


## Pasada 1: píxeles de entrenamiento con los 49 mapas

In [2]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, training_pixels(image_id, background=MODE, features=SUPER)

t = time.time()
samples_all = dict(map_images(pass1, ids, workers=WORKERS))
print(f"Pasada 1: {time.time() - t:.0f} s")

Pasada 1: 524 s


## Ajuste por vuelta (λ elegido dentro del entrenamiento)

In [3]:
def fit_job(job):
    import numpy as np
    from src.pipeline import choose_value, fit_weights, fit_weights_l1
    k, c = job
    cols = COLS[c]
    sc = {i: {key: (v[:, cols] if key in ("x", "x_thr") else v) for key, v in s.items()}
          for i, s in samples_all.items()}
    tr = list(folds.index[folds != k])
    if c == ORDER[0]:
        w, lam, info = fit_weights(sc, tr), None, None
    else:
        w, lam, info = fit_weights_l1(sc, tr, {i: int(folds[i]) for i in tr})
    full = np.zeros(1 + len(SUPER))
    full[0] = w[0]
    full[1 + np.array(cols)] = w[1:]
    return job, full, choose_value(sc, w, tr), lam, info

t = time.time()
jobs = [(k, c) for c in ORDER for k in range(5)]
fit, lams, infos = {}, {}, {}
for job, full, value, lam, info in map_images(fit_job, jobs, workers=WORKERS):
    fit[job], lams[job], infos[job] = (full, value), lam, info
print(f"Ajustes: {time.time() - t:.0f} s")
pd.DataFrame([{"vuelta": k, "candidata": c, "λ": lams[(k, c)], "mapas usados": int(np.count_nonzero(fit[(k, c)][0][1:])),
               "valor fijo": fit[(k, c)][1]} for c in ORDER for k in range(5)]).set_index(["candidata", "vuelta"]).round(5)

Ajustes: 701 s


λ  mapas usados  valor fijo
candidata          vuelta                                   
A. 31 mapas        0           NaN            31     0.88043
                   1           NaN            31     0.87583
                   2           NaN            31     0.87899
                   3           NaN            31     0.86457
                   4           NaN            31     0.87316
B. L1 sobre los 31 0       0.00241            13     0.85811
                   1       0.00243            14     0.85162
                   2       0.00240            15     0.85638
                   3       0.00243            14     0.85759
                   4       0.00243            14     0.86759
C. L1 sobre los 49 0       0.00241            20     0.86007
                   1       0.00243            20     0.87082
                   2       0.00163            21     0.88012
                   3       0.00243            21     0.86059
                   4       0.00165            22     0.87538

In [4]:
print("En cuántas de las 5 vueltas queda cada mapa (peso distinto de cero), y su peso medio:")
kept = pd.DataFrame({c: [sum(fit[(k, c)][0][1 + j] != 0 for k in range(5)) for j in range(len(SUPER))]
                     for c in ORDER[1:]}, index=SUPER)
mean_w = pd.DataFrame({c + " (peso)": [np.mean([fit[(k, c)][0][1 + j] for k in range(5)]) for j in range(len(SUPER))]
                       for c in ORDER}, index=SUPER)
kept.join(mean_w).round(2)

En cuántas de las 5 vueltas queda cada mapa (peso distinto de cero), y su peso medio:


,B. L1 sobre los 31,C. L1 sobre los 49,A. 31 mapas (peso),B. L1 sobre los 31 (peso),C. L1 sobre los 49 (peso)
verde corregido,0,1,-0.52,0.00,-0.00
sombrero de copa,5,5,0.36,0.12,0.18
detector de líneas,0,0,-1.54,0.00,0.00
filtro adaptado,0,0,-0.19,0.00,0.00
Gabor,5,4,1.32,0.37,0.09
Frangi,0,0,2.45,0.00,0.00
detector de líneas 2,0,0,-0.08,0.00,0.00
filtro adaptado 2,0,0,0.03,0.00,0.00
Gabor 2,0,0,-0.03,0.00,0.00
Frangi 2,4,4,-1.17,-0.10,-0.16


## Pasada 2: las personas de afuera

In [5]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import feature_scores
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    rgb = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    auto = detect_fov_robust(rgb)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    probs, _, _ = feature_scores(rgb, [fit[(k, c)][0] for c in ORDER], 2, background=MODE, features=SUPER)
    rows = []
    for c, p in zip(ORDER, probs):
        m = (upscale(p, fov.shape) > fit[(k, c)][1]) & auto
        marked = m & fov
        rows.append({"image": image_id, "candidata": c, "matthews": matthews(m, truth, fov), "finos": m[thin].mean(),
                     "dice_linea_central": centerline_dice(m, truth, fov),
                     "precision": (marked & truth).sum() / max(marked.sum(), 1)})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05m_l1.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

Pasada 2: 522 s


## Resultados

In [6]:
MEAS = ("matthews", "finos", "dice_linea_central", "precision")
rows = []
for c in ORDER[1:]:
    for meas in MEAS:
        w = R.pivot(index="image", columns="candidata", values=meas)
        d = (w[c] - w[ORDER[0]]).to_frame()
        lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
        rows.append({"candidata": c, "medida": meas, "diferencia con A": d.iloc[:, 0].mean(), "inferior": lo[0],
                     "superior": hi[0], "imágenes que mejoran": int((d.iloc[:, 0] > 0).sum())})
table = pd.DataFrame(rows).set_index(["candidata", "medida"])
print("Medias:")
display(R.pivot_table(index="candidata", values=list(MEAS)).loc[ORDER, list(MEAS)].round(4))
table.round(4)

Medias:


,matthews,finos,dice_linea_central,precision
candidata,,,,
A. 31 mapas,0.7527,0.3480,0.7583,0.8019
B. L1 sobre los 31,0.7502,0.3298,0.7523,0.7986
C. L1 sobre los 49,0.7502,0.3437,0.7418,0.8077


diferencia con A  inferior  superior  imágenes que mejoran
candidata          medida                                                                        
B. L1 sobre los 31 matthews                     -0.0025   -0.0050   -0.0003                     8
                   finos                        -0.0182   -0.0258   -0.0099                     6
                   dice_linea_central           -0.0060   -0.0090   -0.0033                     6
                   precision                    -0.0033   -0.0085    0.0009                    10
C. L1 sobre los 49 matthews                     -0.0026   -0.0042   -0.0009                     8
                   finos                        -0.0043   -0.0109    0.0020                    12
                   dice_linea_central           -0.0165   -0.0198   -0.0137                     0
                   precision                     0.0058    0.0013    0.0098                    23

In [7]:
b = table.loc[ORDER[1]]
cond1 = bool(all(b.loc[m, "inferior"] > -C.NONINF_MARGIN_MATTHEWS for m in ("matthews", "finos", "dice_linea_central")))
cond2 = bool(all(np.count_nonzero(fit[(k, ORDER[1])][0][1:]) < len(C.SEGMENT_FEATURES) for k in range(5)))
print("B no inferior (1):", cond1, "| B usa menos mapas en las 5 vueltas (2):", cond2)
print("→ se adopta:", ORDER[1] if cond1 and cond2 else ORDER[0])

B no inferior (1): True | B usa menos mapas en las 5 vueltas (2): True
→ se adopta: B. L1 sobre los 31


In [8]:
print("Por grupo (media):")
R.join(dev["group"], on="image").pivot_table(index="candidata", columns="group",
                                              values=["matthews", "finos", "precision"]).loc[ORDER].round(3)

Por grupo (media):


finos               matthews               precision              
group                  dr      g      h       dr      g      h        dr      g      h
candidata                                                                             
A. 31 mapas         0.363  0.334  0.347    0.708  0.754  0.796     0.716  0.771  0.919
B. L1 sobre los 31  0.355  0.317  0.318    0.701  0.753  0.797     0.705  0.771  0.919
C. L1 sobre los 49  0.349  0.332  0.350    0.705  0.754  0.791     0.723  0.778  0.922

## Decisión (escrita después de correr)

*(pendiente)*